In [1]:
#Install Dependencies
!pip install xgboost scikit-learn pandas numpy joblib

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import joblib
import os
# Create a models directory to save outputs later
os.makedirs('models', exist_ok=True)

In [4]:
print("Loading dataset directly from GitHub...")

# URL to the raw CSV file in your GitHub repo
data_url = "https://raw.githubusercontent.com/shreyascode11/The-Last-CEO/main/corporate_ai_adoption_dataset.csv"
df = pd.read_csv(data_url)

print("Removing extreme outliers (1st to 99th percentile)...")
target_col = 'revenue_impact'
low = df[target_col].quantile(0.01)
high = df[target_col].quantile(0.99)
df = df[(df[target_col] >= low) & (df[target_col] <= high)]
print(f"Dataset size after outlier removal: {len(df)} rows")


df.head()

Loading dataset directly from GitHub...
Removing extreme outliers (1st to 99th percentile)...
Dataset size after outlier removal: 196000 rows


,company_id,industry,country,year,ai_adoption_level,ai_investment_usd,automation_rate,cost_savings,revenue_impact,productivity_gain,employee_ai_training_hours,ai_maturity_score,deployment_count
0,CORP-06613,Financial Services,China,2029,0.4987,11747237,0.4119,3519354,2751513,0.3924,76.8,6.37,29
1,CORP-01597,Agriculture,Germany,2032,0.5213,1267219,0.4580,295244,304776,0.4639,83.2,7.19,37
2,CORP-02938,Energy,United States,2024,0.6147,8168951,0.5821,2472329,5170304,0.4953,123.1,6.72,26
3,CORP-05207,Retail,Germany,2021,0.4401,1234261,0.2880,512061,-233448,0.3078,63.1,5.68,21
4,CORP-07489,Technology,United States,2024,0.1918,5000645,0.1906,2122064,2110644,0.1910,29.6,4.33,16


In [5]:
# Cell 3: Feature Engineering
print("Engineering new interaction features...")

# Standard interaction features
df["investment_per_deployment"] = df["ai_investment_usd"] / (df["deployment_count"]+1)
df["training_per_deployment"] = df["employee_ai_training_hours"] / (df["deployment_count"]+1)
df["investment_maturity"] = df["ai_investment_usd"] * df["ai_maturity_score"]
df["automation_maturity"] = df["automation_rate"] * df["ai_maturity_score"]
df["training_adoption"] = df["employee_ai_training_hours"] * df["ai_adoption_level"]

# Recommended interaction features for tuned model
df["investment_training"] = df["ai_investment_usd"] * df["employee_ai_training_hours"]
df["automation_investment"] = df["automation_rate"] * df["ai_investment_usd"]
df["deployment_training"] = df["deployment_count"] * df["employee_ai_training_hours"]

print("Feature engineering complete.")


Engineering new interaction features...
Feature engineering complete.


In [7]:
# Cell 4: Define Features and Preprocessor
features = [
    'industry', 'country', 'year', 'ai_adoption_level', 'ai_investment_usd',
    'automation_rate', 'employee_ai_training_hours', 'ai_maturity_score', 'deployment_count',
    'investment_per_deployment', 'training_per_deployment', 'investment_maturity',
    'automation_maturity', 'training_adoption',
    'investment_training', 'automation_investment', 'deployment_training'
]

X = df[features]
y_revenue = df[target_col]

numeric_features = [
    'year', 'ai_adoption_level', 'ai_investment_usd', 'automation_rate',
    'employee_ai_training_hours', 'ai_maturity_score', 'deployment_count',
    'investment_per_deployment', 'training_per_deployment', 'investment_maturity',
    'automation_maturity', 'training_adoption',
    'investment_training', 'automation_investment', 'deployment_training'
]
categorical_features = ['industry', 'country']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

print("Preprocessor configured.")


Preprocessor configured.


In [8]:
# Cell 5: Train/Test Split and Transformation
print("Preprocessing data and splitting into train/test sets...")
X_train_rev, X_test_rev, y_train_rev, y_test_rev = train_test_split(X, y_revenue, test_size=0.2, random_state=42)

# Fit on training data, transform both
X_train_rev_transformed = preprocessor.fit_transform(X_train_rev)
X_test_rev_transformed = preprocessor.transform(X_test_rev)

print(f"Training set shape: {X_train_rev_transformed.shape}")
print(f"Testing set shape: {X_test_rev_transformed.shape}")


Preprocessing data and splitting into train/test sets...
Training set shape: (156800, 40)
Testing set shape: (39200, 40)


In [9]:
# Cell 6: Train Tuned XGBoost Model
print("Training Revenue Impact Model with Advanced XGBoost Parameters...")

xgb_revenue = xgb.XGBRegressor(
    n_estimators=1000,          # Increased trees, let early stopping decide when to stop
    learning_rate=0.01,         # Slower learning rate for finer adjustments
    max_depth=8,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1,
    early_stopping_rounds=50,   # Stops training if eval metrics don't improve
    random_state=42,
    n_jobs=-1
)

# Train the model
xgb_revenue.fit(
    X_train_rev_transformed,
    y_train_rev,
    eval_set=[(X_test_rev_transformed, y_test_rev)],
    verbose=True  # Set to True so you can watch the training progress in Colab
)
print("\nTraining Complete.")


Training Revenue Impact Model with Advanced XGBoost Parameters...
[0]	validation_0-rmse:2814546.26384
[1]	validation_0-rmse:2795130.09239
[2]	validation_0-rmse:2775901.20341
[3]	validation_0-rmse:2756886.61003
[4]	validation_0-rmse:2738158.81847
[5]	validation_0-rmse:2720455.56872
[6]	validation_0-rmse:2702200.03334
[7]	validation_0-rmse:2684197.93448
[8]	validation_0-rmse:2666479.15074
[9]	validation_0-rmse:2649666.55441
[10]	validation_0-rmse:2632452.18315
[11]	validation_0-rmse:2616536.05080
[12]	validation_0-rmse:2599735.08481
[13]	validation_0-rmse:2583877.98895
[14]	validation_0-rmse:2567422.42334
[15]	validation_0-rmse:2551878.46600
[16]	validation_0-rmse:2535882.17016
[17]	validation_0-rmse:2520949.45161
[18]	validation_0-rmse:2505388.61061
[19]	validation_0-rmse:2490095.55940
[20]	validation_0-rmse:2474935.45140
[21]	validation_0-rmse:2459988.33075
[22]	validation_0-rmse:2445283.43549
[23]	validation_0-rmse:2430766.80122
[24]	validation_0-rmse:2416508.23302
[25]	validation_0-r

In [10]:
# Cell 7: Evaluate Model and Save
print("\n--- Model Evaluation ---")
y_pred_rev = xgb_revenue.predict(X_test_rev_transformed)

mae = mean_absolute_error(y_test_rev, y_pred_rev)
rmse = np.sqrt(mean_squared_error(y_test_rev, y_pred_rev))
r2 = r2_score(y_test_rev, y_pred_rev)

print(f"New Revenue Impact Model Metrics:")
print(f"  MAE:  {mae:,.2f}")
print(f"  RMSE: {rmse:,.2f}")
print(f"  R2:   {r2:.4f}")

old_r2 = 0.5957
if r2 > old_r2:
    print(f"\n[SUCCESS] Performance improved from R2 {old_r2:.4f} to {r2:.4f}!")

    # Save to the Colab environment
    save_path = os.path.join('models', 'revenue_model.joblib')
    print(f"Saving new model to '{save_path}'...")
    revenue_package = {'preprocessor': preprocessor, 'model': xgb_revenue}
    joblib.dump(revenue_package, save_path)
    print("New model saved successfully. You can download it from the Colab file explorer.")
else:
    print(f"\n[FAILED] Performance did not improve significantly (Current R2: {r2:.4f} vs Old R2: {old_r2:.4f}).")



--- Model Evaluation ---
New Revenue Impact Model Metrics:
  MAE:  842,399.06
  RMSE: 1,554,296.41
  R2:   0.6993

[SUCCESS] Performance improved from R2 0.5957 to 0.6993!
Saving new model to 'models/revenue_model.joblib'...
New model saved successfully. You can download it from the Colab file explorer.
